# Notebook 02 — Inspecting the Buddy Jr Environment

Before training any agent it pays to spend time **understanding the environment
you are working with**.  This notebook answers four questions:

1. What does the observation space look like?
2. What does the action space look like?
3. What actually happens when you call `reset()` and `step()`?
4. How does the arm's forward kinematics map joint angles to tip position?

No prior knowledge is required; the environment uses the fast default
`KinematicBackend` (no PyBullet/MuJoCo needed) so every cell runs on CPU.

---

## 1. Imports and environment creation

Importing `rl_lab` triggers `register_envs()`, which registers
`BuddyJrReach-v0`, `BuddyJrReachDiscrete-v0`, and `BuddyJrCameraPoint-v0`
with Gymnasium so `gym.make()` can find them.

In [ ]:
import numpy as np
import gymnasium as gym
import rl_lab  # registers all env ids

print('rl_lab version:', rl_lab.__version__)
print('gym version   :', gym.__version__)

## 2. Create the discrete environment

`BuddyJrReachDiscrete-v0` wraps the same reach task but uses a
`Discrete(9)` action space.  This makes it compatible with tabular methods
(Q-learning, SARSA) and DQN, which expect integer actions.

The 9 actions are:
- 0 — no-op (hold all joints)
- 1/2 — nudge `base_yaw` +/-
- 3/4 — nudge `shoulder_pitch` +/-
- 5/6 — nudge `elbow_pitch` +/-
- 7/8 — nudge `camera_tilt` +/-

In [ ]:
env = gym.make('BuddyJrReachDiscrete-v0')
print('Environment id  :', env.spec.id)
print('Max episode steps:', env.spec.max_episode_steps)

## 3. Observation space

In [ ]:
obs_space = env.observation_space
print('Type            :', type(obs_space).__name__)
print('Shape           :', obs_space.shape)
print('dtype           :', obs_space.dtype)
print('Low  (first 4)  :', obs_space.low[:4])
print('High (first 4)  :', obs_space.high[:4])

The observation is a **17-element float32 vector** with all values in
``[-1, 1]``.  The 17 elements break down as:

| Index | Content | Note |
|-------|---------|------|
| 0-3   | `sin(q)` — sine of the 4 joint angles | avoids ±π wrap |
| 4-7   | `cos(q)` — cosine of the 4 joint angles | paired with sin |
| 8-10  | camera-tip position (xyz), scaled | multiply by 1/4.5 for metres |
| 11-13 | target position (xyz), scaled | same scale |
| 14-16 | vector from tip to target, scaled | direction + magnitude hint |

## 4. Action space

In [ ]:
act_space = env.action_space
print('Type            :', type(act_space).__name__)
print('n (num actions) :', act_space.n)
print('Sample actions  :', [env.action_space.sample() for _ in range(6)])

## 5. Reset the environment

In [ ]:
obs, info = env.reset(seed=7)

print('obs shape   :', obs.shape)
print('obs dtype   :', obs.dtype)
print('obs (raw)   :', np.round(obs, 3))
print()
print('info keys   :', list(info.keys()))
print('distance    :', round(info['distance'], 4), 'm  (tip-to-target)')
print('joint_q     :', np.round(info['joint_q'], 3), 'rad')
print('ee_pos      :', np.round(info['ee_pos'], 4), 'm')
print('target      :', np.round(info['target'], 4), 'm')

## 6. Decode the observation vector

The raw numbers are not very readable.  This helper unpacks them into human
form.

In [ ]:
POS_SCALE = 4.5  # matches rl_lab.env.spaces.POS_SCALE
JOINT_NAMES = ['base_yaw', 'shoulder_pitch', 'elbow_pitch', 'camera_tilt']

def decode_obs(obs):
    """Print a human-readable breakdown of a 17-D Buddy Jr observation."""
    sin_q   = obs[0:4]
    cos_q   = obs[4:8]
    ee_pos  = obs[8:11]  / POS_SCALE
    tgt_pos = obs[11:14] / POS_SCALE
    tip2tgt = obs[14:17] / POS_SCALE
    q_rad   = np.arctan2(sin_q, cos_q)
    q_deg   = np.degrees(q_rad)

    print('--- Joint angles (recovered from sin/cos) ---')
    for name, rad, deg in zip(JOINT_NAMES, q_rad, q_deg):
        print(f'  {name:<16} {rad:>7.3f} rad  ({deg:>6.1f} deg)')
    print(f'--- Positions (metres) ---')
    print(f'  camera-tip  x={ee_pos[0]:.4f}  y={ee_pos[1]:.4f}  z={ee_pos[2]:.4f}')
    print(f'  target      x={tgt_pos[0]:.4f}  y={tgt_pos[1]:.4f}  z={tgt_pos[2]:.4f}')
    dist = np.linalg.norm(tip2tgt)
    print(f'  tip→target  dist={dist:.4f} m')

decode_obs(obs)

## 7. Stepping through the environment

The `step(action)` call returns `(obs, reward, terminated, truncated, info)`.
- `terminated = True` when the tip is within 2 cm of the target (success).
- `truncated = True` when the episode hits the 200-step limit.

In [ ]:
obs, info = env.reset(seed=42)

print(f'{"Step":>5}  {"Action":>6}  {"Reward":>8}  {"Distance":>10}  {"Done?":>6}')
print('-' * 46)

for step_i in range(10):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    print(f'{step_i+1:>5}  {action:>6}  {reward:>8.4f}  {info["distance"]:>10.4f}  {str(done):>6}')
    if done:
        print('  Episode ended early.')
        break

## 8. Run a short random episode and collect statistics

In [ ]:
obs, info = env.reset(seed=0)
distances = [info['distance']]
rewards_list = []

for _ in range(50):
    obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
    distances.append(info['distance'])
    rewards_list.append(reward)
    if terminated or truncated:
        break

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3))
ax1.plot(distances, color='steelblue', lw=1.5)
ax1.axhline(0.02, color='green', ls='--', lw=1, label='success threshold (2 cm)')
ax1.set_xlabel('Step')
ax1.set_ylabel('Distance to target (m)')
ax1.set_title('Random policy — tip-to-target distance')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

ax2.plot(rewards_list, color='coral', lw=1.5)
ax2.axhline(0, color='grey', ls=':', lw=0.8)
ax2.set_xlabel('Step')
ax2.set_ylabel('Reward')
ax2.set_title('Random policy — per-step reward')
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.savefig('/tmp/random_episode.png', dpi=110)
plt.close(fig)
print(f'Steps taken  : {len(rewards_list)}')
print(f'Total reward : {sum(rewards_list):.3f}')
print(f'Final dist   : {distances[-1]:.4f} m')
print('Plot saved to /tmp/random_episode.png')

## 9. Forward kinematics

`rl_lab.robot.kinematics.forward(q)` computes the exact 3-D position of the
camera tip from the 4 joint angles.  No physics engine is needed — it is pure
matrix multiplication (SE3 transforms).

The joint vector `q = [base_yaw, shoulder_pitch, elbow_pitch, camera_tilt]`
is in radians.  A neutral pose is `q = [0, 0, 0, 0]`.

In [ ]:
from rl_lab.robot.kinematics import forward, forward_position

# Neutral pose
q_neutral = np.zeros(4)
pose = forward(q_neutral)
print('Neutral pose (q = 0, 0, 0, 0):')
print(f'  tip position   : {np.round(pose.position, 4)} m')
print(f'  tip orientation: {np.round(pose.orientation, 4)}  (xyzw quaternion)')


In [ ]:
# Try a few named poses
poses = {
    'neutral'          : [0.0,  0.0,  0.0,  0.0],
    'arm_raised'       : [0.0,  0.8, -0.8,  0.0],
    'yawed_right_45deg': [0.785, 0.0, 0.0,  0.0],
    'reaching_forward' : [0.0,  0.5, -0.5,  0.0],
}

print(f'{"pose name":<22}  {"x":>7}  {"y":>7}  {"z":>7}  (metres)')
print('-' * 52)
for name, q in poses.items():
    p = forward_position(np.array(q))
    print(f'{name:<22}  {p[0]:>7.4f}  {p[1]:>7.4f}  {p[2]:>7.4f}')

## 10. FK matches the env's end-effector position

The environment uses the same FK under the hood.  Verify that the `info['ee_pos']`
returned after `reset()` agrees with `forward_position(info['joint_q'])`.

In [ ]:
obs, info = env.reset(seed=99)
q_from_env  = info['joint_q']
ee_from_env = info['ee_pos']
ee_from_fk  = forward_position(q_from_env)

print('ee_pos from env :', np.round(ee_from_env, 6))
print('ee_pos from FK  :', np.round(ee_from_fk,  6))
diff = np.linalg.norm(ee_from_env - ee_from_fk)
print(f'max difference  : {diff:.2e} m  (should be < 1e-12 for kinematic backend)')

## 11. Visualise the arm's workspace

Sample random reachable joint angles and plot where the tip ends up.

In [ ]:
from rl_lab.robot.buddy_jr import JOINT_LIMIT  # = pi/2 rad

rng = np.random.default_rng(0)
N = 800
Q = rng.uniform(-JOINT_LIMIT, JOINT_LIMIT, size=(N, 4))
tips = np.stack([forward_position(q) for q in Q])  # (N, 3)

fig = plt.figure(figsize=(7, 5))
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(tips[:, 0], tips[:, 1], tips[:, 2],
                c=tips[:, 2], cmap='viridis', s=6, alpha=0.5)
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_zlabel('Z (m)')
ax.set_title('Buddy Jr camera-tip workspace (random joint samples)')
fig.colorbar(sc, ax=ax, label='Z height (m)', shrink=0.6)
fig.tight_layout()
plt.savefig('/tmp/workspace.png', dpi=110)
plt.close(fig)
print(f'Workspace extents:')
print(f'  X: [{tips[:,0].min():.3f}, {tips[:,0].max():.3f}] m')
print(f'  Y: [{tips[:,1].min():.3f}, {tips[:,1].max():.3f}] m')
print(f'  Z: [{tips[:,2].min():.3f}, {tips[:,2].max():.3f}] m')
print('Plot saved to /tmp/workspace.png')

## 12. Clean up and next steps

In [ ]:
env.close()
print('Environment closed.')
print()
print('Next steps:')
print('  * Run experiments/03_qlearning.py to train a tabular Q-learning agent.')
print('  * Run experiments/05_dqn.py to train a DQN agent on BuddyJrReachDiscrete-v0.')
print('  * Open notebooks/99_analysis_template.ipynb to plot a saved learning curve.')